# 04 — Hyper-parameter Tuning  (D4 Level 2)

**NovaFin Group Capstone · ePGD MLDS, IIIT Bombay · Group 2 · Dipesh Kumar Yadav**

Level 2 of the fine-tuning ladder: Optuna, with **pruning**, a **written
search-space rationale** for all 85 tunable parameters, and **study persistence
so runs resume**.

### Why TPE rather than grid or random search

The credit LightGBM space has 9 axes. A grid of only 4 values each is
**262,144 fits**. Random search is unbiased but memoryless — it learns nothing
from the 40 trials it already ran. Tree-structured Parzen Estimation models
P(params | good) against P(params | bad) and samples where the ratio is high,
which is what finds the `num_leaves` ↔ `min_child_samples` interaction the
spaces are explicitly built around.

> Bergstra, Bardenet, Bengio & Kégl (2011), *Algorithms for hyper-parameter
> optimization*, NeurIPS 24.

### The pruning decision that matters

Optuna prunes by comparing a trial's value at step *k* against other trials at
step *k*. **If step *k* is fold 1, a trial dies on one fold's evidence.**
Phase 2 measured per-fold AUC standard errors of **0.096** (initiatives) and
**0.070** (churn) — fold-1 noise alone is wider than the gap between a good and
a bad configuration.

So `SafeMedianPruner` enforces `min_folds_before_prune`: **2** by default,
**3** on the two low-power modules. A hopeless trial still dies after 3 of 50
fits; it just cannot be killed by noise.

> Prerequisites: notebooks `00`–`03`.

## 1 · Preamble

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

IN_COLAB = "google.colab" in str(get_ipython())  # noqa: F821
if IN_COLAB:
    import subprocess, sys
    from pathlib import Path

    REPO_URL = "https://github.com/yadavdipesh/novafin-capstone.git"
    if not Path("/content/novafin-capstone").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/novafin-capstone"], check=True)
    os.chdir("/content/novafin-capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["NOVAFIN_DATA_RAW"] = (
        "/content/drive/MyDrive/01 ML in Finance/data"
    )
    # PUT THE STUDY DATABASE ON DRIVE. This is the whole point of Level 2
    # persistence: a free-tier session dies at ~12 h or ~90 min idle, and a
    # study on the VM's local disk dies with it.
    os.makedirs("/content/drive/MyDrive/novafin_studies", exist_ok=True)
    os.environ["NOVAFIN_OPTUNA_STORAGE"] = (
        "/content/drive/MyDrive/novafin_studies/novafin_studies.db"
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from novafin.config import load_config
from novafin.data import load_all, make_feature_frame, make_splitter, holdout_by_time
from novafin.features import build_features
from novafin.models import load_model_specs, train_module
from novafin.models.tune import (
    load_search_spaces, load_study, study_summary, tune_model, validate_search_space,
)
from novafin.tracking import ExperimentTracker
from novafin.utils.logging_utils import setup_logging
from novafin.utils.seed import seed_everything
from novafin.utils.theme import apply_theme, color, save_figure, semantic_color

cfg = load_config()
setup_logging("WARNING", log_file=cfg.paths.logs / "04_tuning.log")
seed_everything(cfg.reproducibility.seed)
apply_theme()
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

tracker = ExperimentTracker(cfg)
print("config fingerprint:", cfg.fingerprint())
print("study database    :", cfg.paths.optuna_storage)
print("tracking backend  :", tracker.backend)

## 2 · The search spaces, and why every bound is what it is

This section **is** the D4 "search-space rationale" deliverable. Every one of
the 85 tunable parameters carries a written justification, and
`validate_search_space` rejects any parameter without one — so the discipline
is enforced by the test suite, not merely intended.

Two rules run through all of it:

1. **Bounds derive from data size, not convention.** 180 rows cannot fill 31
   leaves; 30,000 rows make a 5-row leaf pure noise-fitting.
2. **Log scale wherever the parameter acts multiplicatively.** Sampling
   `learning_rate` uniformly over [0.01, 0.3] spends two-thirds of the trials
   above 0.1 — the wrong end for small data.

In [ ]:
import yaml
from pathlib import Path

raw = yaml.safe_load(Path("configs/search_spaces.yaml").read_text())
modules = [k for k in raw if k != "defaults"]

rows = []
for module in modules:
    spaces, settings = load_search_spaces(module)
    for name, space in spaces.items():
        problems = validate_search_space(space)
        rows.append({
            "module": module, "model": name, "n_params": len(space.params),
            "objective": settings.objective,
            "direction": "min" if settings.minimise else "max",
            "n_trials": settings.n_trials,
            "min_folds_before_prune": settings.min_folds_before_prune,
            "valid": "OK" if not problems else f"FAIL {problems}",
        })

overview = pd.DataFrame(rows)
display(overview)
print(f"\ntotal tunable parameters: {overview['n_params'].sum()}")
assert (overview["valid"] == "OK").all(), "a search space failed validation"

In [ ]:
# The rationale table for one model - this goes verbatim into the D7 guide.
spaces_loans, settings_loans = load_search_spaces("loans")
print(f"M2 credit risk - objective '{settings_loans.objective}', "
      f"{settings_loans.n_trials} trials\n")
display(spaces_loans["lightgbm"].rationale_table())

In [ ]:
# Contrast the capacity ceilings. The SAME parameter, three sample sizes.
comparison = []
for module in ("initiatives", "loans", "transactions", "market"):
    spaces, _ = load_search_spaces(module)
    if "lightgbm" not in spaces:
        continue
    params = spaces["lightgbm"].params
    n_rows = cfg.dataset(module).expected_rows
    leaves_high = params["num_leaves"]["high"]
    comparison.append({
        "module": module, "rows": n_rows,
        "num_leaves_high": leaves_high,
        "rows_per_leaf": round(n_rows / leaves_high, 1),
        "min_child_samples_low": params.get("min_child_samples", {}).get("low"),
    })
display(pd.DataFrame(comparison))
print("The ceiling tracks the sample size. A single copied search space would")
print("either cripple the fraud model or let the initiatives model memorise 180 rows.")

## 3 · Baselines to beat

Level 2 is only meaningful against Level 1. These are the numbers from
notebook `03`, re-read here so the improvement is measured, not assumed.

In [ ]:
results = load_all(cfg=cfg)
features, matrices = {}, {}
for key, loaded in results.items():
    built = build_features(key, loaded.frame, cfg)
    features[key] = built
    spec = cfg.dataset(key)
    extra = [c for c in ("fwd_return_5d", "fwd_inflows_5d") if c in built.frame.columns]
    X, y = make_feature_frame(built.frame, spec, extra_drop=extra)
    X = X.drop(columns=[c for c in X.columns
                        if pd.api.types.is_datetime64_any_dtype(X[c])], errors="ignore")
    matrices[key] = (X, built.frame[built.target])

baseline_path = cfg.paths.tables / "03_baseline_summary.csv"
if baseline_path.exists():
    baselines = pd.read_csv(baseline_path)
    display(baselines)
else:
    print("Run notebook 03 first - Level 2 improvement is measured against Level 1.")
    baselines = pd.DataFrame()

## 4 · M2 Credit Risk — the worked tuning example

Objective **KS**. Brier is reported alongside, because PD feeds
`ECL = PD × LGD × EAD` and a mis-calibrated PD produces the wrong money — a
tuned model that wins on KS while losing on Brier is not obviously an
improvement, and the table below makes that visible.

In [ ]:
X_loans, y_loans = matrices["loans"]
splitter_loans, kwargs_loans = make_splitter("loans", features["loans"].frame, cfg=cfg)

baseline_ks = float("nan")
if len(baselines):
    match = baselines[(baselines["module"] == "loans") & (baselines["model"] == "lightgbm")]
    if len(match) and "ks" in match.columns:
        baseline_ks = float(match["ks"].iloc[0])
print(f"Level-1 baseline KS (lightgbm): {baseline_ks}")

result_loans = tune_model(
    "loans", "lightgbm", X_loans, y_loans, splitter_loans,
    task="binary_classification", cfg=cfg, split_kwargs=kwargs_loans,
    baseline_value=baseline_ks, show_progress=True,
)
display(study_summary([result_loans]))
print("\nbest parameters:")
for name, value in result_loans.best_params.items():
    print(f"   {name:<22} {value}")

In [ ]:
# Optimisation history - and how much compute pruning saved.
trials = result_loans.trials
if trials is not None and len(trials):
    completed = trials[trials["state"] == "COMPLETE"].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

    axes[0].scatter(completed["number"], completed["value"], s=22,
                    color=color("teal"), label="completed")
    running_best = completed["value"].cummax()
    axes[0].plot(completed["number"], running_best, color=color("navy"), lw=2,
                 label="best so far")
    if np.isfinite(baseline_ks):
        axes[0].axhline(baseline_ks, ls="--", lw=1.2,
                        color=semantic_color("critical"), label="Level-1 baseline")
    axes[0].set_xlabel("Trial"); axes[0].set_ylabel(result_loans.objective.upper())
    axes[0].set_title("Optimisation history"); axes[0].legend()

    states = trials["state"].value_counts()
    axes[1].bar(states.index, states.values,
                color=[color("teal") if s == "COMPLETE" else semantic_color("warning")
                       for s in states.index])
    axes[1].set_title("Trial outcomes"); axes[1].set_ylabel("Count")
    plt.tight_layout()
    save_figure(fig, "04_m02_optimisation_history", close=False)
    plt.show()

    n_folds = int(cfg.splits("loans").get("n_splits", 5))
    min_folds = settings_loans.min_folds_before_prune
    fits_saved = result_loans.n_pruned * max(0, n_folds - min_folds)
    print(f"pruned trials    : {result_loans.n_pruned} of {result_loans.n_trials}")
    print(f"model fits saved : ~{fits_saved} (a pruned trial stops after "
          f"{min_folds} of {n_folds} folds)")

### 4.1 · Does the tuned model actually win?

A single KS number is not enough. Re-running the full evaluation lets us check
that discrimination improved **without** calibration getting worse — the
trade-off that matters when the output is multiplied by money.

In [ ]:
from novafin.models import ModelSpec, cross_validate_model, evaluate_cv

catalog_loans = load_model_specs("loans")
base_spec = catalog_loans.get("lightgbm")
tuned_spec = ModelSpec(
    name="lightgbm_tuned", class_path=base_spec.class_path,
    params={**base_spec.params, **result_loans.best_params}, scale=base_spec.scale,
)

comparison_rows = []
for label, spec in [("L1 baseline", base_spec), ("L2 tuned", tuned_spec)]:
    cv = cross_validate_model(spec, X_loans, y_loans, splitter_loans,
                              task="binary_classification", module="loans", cfg=cfg,
                              split_kwargs=kwargs_loans, defaults=catalog_loans.defaults)
    evaluation = evaluate_cv(cv, y_loans, cfg=cfg)
    ece = float("nan")
    if evaluation.calibration is not None and not evaluation.calibration.empty:
        weights = evaluation.calibration["n"] / evaluation.calibration["n"].sum()
        ece = float((weights * evaluation.calibration["gap"].abs()).sum())
    comparison_rows.append({
        "level": label, "ks": evaluation.metrics.get("ks"),
        "roc_auc": evaluation.metrics.get("roc_auc"),
        "pr_auc": evaluation.metrics.get("pr_auc"),
        "brier": evaluation.metrics.get("brier"), "ece": ece,
        "ks_fold_std": evaluation.metrics.get("cv_ks_std"),
    })

comparison = pd.DataFrame(comparison_rows).round(5)
display(comparison)

delta_ks = comparison["ks"].iloc[1] - comparison["ks"].iloc[0]
fold_std = comparison["ks_fold_std"].iloc[1]
print(f"\nKS improvement       : {delta_ks:+.5f}")
print(f"fold-to-fold KS std  : {fold_std:.5f}")
print("\nIF the improvement is smaller than the fold standard deviation, say so.")
print("Tuning that moves a metric by less than its own noise is not an improvement,")
print("and reporting it as one is exactly what a viva will probe.")

## 5 · M3 Fraud — tuning toward PR-AUC, deciding on cost

The objective is PR-AUC because ROC-AUC flatters at a 2.28% base rate. The
**decision** is still the threshold, and it is re-optimised after tuning — a
better-ranked model shifts the cost-optimal operating point.

In [ ]:
X_txn, y_txn = matrices["transactions"]
txn_frame = features["transactions"].frame
train_idx, holdout_idx = holdout_by_time(txn_frame, "transactions", cfg=cfg)
splitter_txn, _ = make_splitter("transactions", txn_frame, cfg=cfg)

baseline_pr = float("nan")
if len(baselines):
    match = baselines[(baselines["module"] == "transactions") & (baselines["model"] == "lightgbm")]
    if len(match) and "pr_auc" in match.columns:
        baseline_pr = float(match["pr_auc"].iloc[0])

result_txn = tune_model(
    "transactions", "lightgbm", X_txn.iloc[train_idx], y_txn.iloc[train_idx],
    splitter_txn, task="binary_classification", cfg=cfg,
    baseline_value=baseline_pr, show_progress=True,
)
display(study_summary([result_txn]))
print("best parameters:")
for name, value in result_txn.best_params.items():
    print(f"   {name:<22} {value}")

weight = result_txn.best_params.get("scale_pos_weight")
if weight is not None:
    full_balance = (1 - 0.0228) / 0.0228
    print(f"\nscale_pos_weight chosen: {weight:.2f}  (full inverse balance is {full_balance:.1f})")
    print("An optimum well below full balance is common and informative: it means")
    print("the model prefers calibrated probabilities over forced recall, which is")
    print("the right preference when a cost threshold is applied afterwards.")

In [ ]:
# Re-optimise the operating point for the tuned model.
from novafin.evaluate import optimal_threshold

catalog_txn = load_model_specs("transactions")
base_txn = catalog_txn.get("lightgbm")
tuned_txn = ModelSpec(name="lightgbm_tuned", class_path=base_txn.class_path,
                      params={**base_txn.params, **result_txn.best_params},
                      scale=base_txn.scale)

cost_fn = cfg.fin("fraud", "cost_missed_fraud_inr", default=10000)
cost_fp = cfg.fin("fraud", "cost_false_positive_inr", default=500)

cost_rows = []
for label, spec in [("L1 baseline", base_txn), ("L2 tuned", tuned_txn)]:
    cv = cross_validate_model(spec, X_txn.iloc[train_idx], y_txn.iloc[train_idx],
                              splitter_txn, task="binary_classification",
                              module="transactions", cfg=cfg, defaults=catalog_txn.defaults)
    mask = cv.oof_mask
    best = optimal_threshold(
        np.asarray(y_txn.iloc[train_idx])[mask],
        np.asarray(cv.oof_predictions)[mask],
        cost_false_negative=cost_fn, cost_false_positive=cost_fp,
    )
    cost_rows.append({"level": label, **{k: round(v, 2) for k, v in best.items()}})

cost_comparison = pd.DataFrame(cost_rows)
display(cost_comparison[["level", "threshold", "total_cost", "n_flagged",
                         "precision", "recall", "saving_vs_best_baseline"]])
delta_cost = cost_comparison["total_cost"].iloc[0] - cost_comparison["total_cost"].iloc[1]
print(f"\nExpected cost reduction from tuning: {delta_cost:,.0f} INR")
print("This is the number the Board cares about - not PR-AUC.")

## 6 · M5/M6 Equity — where regularisation is the whole game

Objective **mean IC**. The signal-to-noise ratio in daily equity returns is
tiny: a model with real capacity fits noise and produces a **negative**
out-of-sample IC. Watch where the optimiser lands on `reg_lambda` and
`num_leaves` — if it pushes to the regularised end of both, that is the data
telling you something, and it belongs in the report.

In [ ]:
X_mkt, y_mkt = matrices["market"]
market_frame = features["market"].frame
splitter_mkt, kwargs_mkt = make_splitter("market", market_frame, cfg=cfg)

baseline_ic = float("nan")
if len(baselines):
    match = baselines[(baselines["module"] == "market") & (baselines["model"] == "lightgbm")]
    if len(match) and "mean_ic" in match.columns:
        baseline_ic = float(match["mean_ic"].iloc[0])

result_mkt = tune_model(
    "market", "lightgbm", X_mkt, y_mkt, splitter_mkt,
    task="panel_regression", cfg=cfg, split_kwargs=kwargs_mkt,
    ic_groups=market_frame["Date"], baseline_value=baseline_ic, show_progress=True,
)
display(study_summary([result_mkt]))

spaces_mkt, _ = load_search_spaces("market")
bounds = spaces_mkt["lightgbm"].params
print("\nWhere the optimiser landed, relative to each range:")
for name, value in result_mkt.best_params.items():
    spec = bounds.get(name, {})
    if spec.get("type") in {"float", "int"}:
        low, high = spec["low"], spec["high"]
        position = (np.log(value / low) / np.log(high / low)) if spec.get("log") else (value - low) / (high - low)
        marker = "REGULARISED end" if (name in {"reg_lambda"} and position > 0.6) or \
                                      (name in {"num_leaves", "n_estimators", "learning_rate"} and position < 0.4) else ""
        print(f"   {name:<20} {value!s:>10}   {position:5.0%} along [{low}, {high}]  {marker}")

In [ ]:
# Ridge on the same target, for contrast. On a low-signal problem a heavily
# shrunk linear model is a serious competitor, and sometimes the winner.
result_ridge = tune_model(
    "market", "ridge", X_mkt, y_mkt, splitter_mkt,
    task="panel_regression", cfg=cfg, split_kwargs=kwargs_mkt,
    ic_groups=market_frame["Date"], baseline_value=baseline_ic,
)
display(study_summary([result_mkt, result_ridge]))
alpha = result_ridge.best_params.get("alpha")
if alpha is not None:
    print(f"best Ridge alpha: {alpha:.3f}  (range was [0.1, 1000])")
    print("A very large alpha means near-total shrinkage genuinely beat a fitted")
    print("model - which is a legitimate and publishable finding on daily returns.")

## 7 · M4 Churn — tuned to demonstrate a fair attempt, not to find a winner

The study is **deliberately short** (30 trials). With 89 positives, running 500
trials would eventually surface a configuration with a flattering CV score, and
that score would be selection noise.

Compare the tuned result against the permutation-test null from notebook `03`.
If the tuned score still sits inside the null distribution, finding **N-01**
stands — and that is the honest, defensible result.

In [ ]:
X_cust, y_cust = matrices["customers"]
splitter_cust, _ = make_splitter("customers", features["customers"].frame, cfg=cfg)
_, settings_cust = load_search_spaces("customers")
print("study note from config:")
print("   " + " ".join(settings_cust.note.split()))

baseline_churn = float("nan")
if len(baselines):
    match = baselines[(baselines["module"] == "customers") & (baselines["model"] == "lightgbm")]
    if len(match) and "pr_auc" in match.columns:
        baseline_churn = float(match["pr_auc"].iloc[0])

result_cust = tune_model(
    "customers", "lightgbm", X_cust, y_cust, splitter_cust,
    task="binary_classification", cfg=cfg, baseline_value=baseline_churn,
)
display(study_summary([result_cust]))
print("\nCompare this against the permutation null in notebook 03.")
print("A tuned PR-AUC inside the null distribution means finding N-01 stands:")
print("there is no learnable churn signal in this dataset, and the 1,000-customer")
print("question is answered on VALUE instead.")

## 8 · Persistence — the Colab-disconnect insurance

This is the requirement that makes Level 2 usable on free-tier Colab. The study
lives in SQLite; `load_if_exists=True` means re-running a cell after a
disconnect **continues from trial *n*** rather than restarting.

Point `NOVAFIN_OPTUNA_STORAGE` at Drive (cell 1 does) and the study survives the
VM entirely.

In [ ]:
study = load_study(result_loans.study_name, cfg)
print(f"study name    : {study.study_name}")
print(f"storage       : {result_loans.storage}")
print(f"trials stored : {len(study.trials)}")
print(f"best value    : {study.best_value:.5f}")

print("\nRe-running the tune_model cell above adds trials to THIS study.")
print("Nothing is recomputed, and a disconnect costs you only the trial in flight.")

db = cfg.paths.optuna_storage
if db.exists():
    print(f"\ndatabase file : {db}  ({db.stat().st_size / 1024:.0f} KB)")
    print("Copy this file to Drive to carry a study between sessions.")

In [ ]:
# Every study in the store, with its progress. Run this after a reconnect
# to see what has already been done.
import sqlite3

if cfg.paths.optuna_storage.exists():
    try:
        import optuna
        summaries = optuna.get_all_study_summaries(f"sqlite:///{cfg.paths.optuna_storage}")
        display(pd.DataFrame([
            {"study": s.study_name, "trials": s.n_trials,
             "direction": s.direction.name if hasattr(s.direction, "name") else str(s.direction),
             "best": getattr(s.best_trial, "value", None)}
            for s in summaries
        ]))
    except Exception as exc:
        print("Could not enumerate studies:", exc)

## 9 · Level-2 summary and persistence of the tuned configurations

The tuned parameters are written to `configs/tuning/` as YAML. That matters for
what comes next: **D4 Level 4** launches new campaigns by editing YAML only, and
these files are the starting point it reads.

In [ ]:
all_tuned = [result_loans, result_txn, result_mkt, result_ridge, result_cust]
summary = study_summary(all_tuned)
display(summary)

summary.to_csv(cfg.paths.tables / "04_tuning_summary.csv", index=False)

# Write the winning configurations back out as YAML.
tuned_dir = cfg.paths.artifacts.parent / "configs" / "tuning"
tuned_dir.mkdir(parents=True, exist_ok=True)
payload = {
    r.module: {r.model_name: {"params": r.best_params,
                              "objective": r.objective,
                              "value": float(r.best_value)}}
    for r in all_tuned if r.best_params
}
target = tuned_dir / "level2_best_params.yaml"
target.write_text(yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")
print(f"\nwritten: {target}")
print(target.read_text()[:1200])

In [ ]:
# Rationale tables for the D7 guide - one CSV per module.
for module in modules:
    spaces, _ = load_search_spaces(module)
    frames = []
    for name, space in spaces.items():
        table = space.rationale_table()
        table.insert(0, "model", name)
        frames.append(table)
    if frames:
        pd.concat(frames, ignore_index=True).to_csv(
            cfg.paths.tables / f"04_search_space_{module}.csv", index=False
        )

print("Tables written:")
for path in sorted(cfg.paths.tables.glob("04_*.csv")):
    print("   ", path.name)

---

## Phase 5 summary

| Guarantee | Established by |
|---|---|
| Every range is justified | 85 parameters, each with a `why`; `validate_search_space` fails the build without one |
| Pruning cannot act on noise | `SafeMedianPruner` with `min_folds_before_prune` (3 on the low-power modules) |
| The objective cannot leak | `_assert_objective_is_oof` checks the out-of-fold mask every trial |
| A disconnect costs one trial | SQLite study, `load_if_exists=True`, database on Drive |
| Improvement is measured, not assumed | every study carries its Level-1 baseline and reports the delta |
| A win on the metric is not automatically a win | KS **and** Brier/ECE reported side by side |

**NEXT:** `05_fine_tuning` — D4 **Level 3**: warm-start / staged boosting for the
tabular modules (`init_model`), layer-wise unfreezing with discriminative
learning rates for the HFT sequence net, and the self-supervised FT-Transformer
+ **LoRA adapter per module head** that makes PEFT genuinely first-class rather
than decorative.